# FieldFind — offline hybrid document retrieval prototype

Pipeline: **Markdown corpus → section chunks → BM25 + dense index → RRF fusion → document grouping → confidence gate → evaluation**.

- Runs fully on CPU. Only `numpy`, `pandas`, `scikit-learn` are required.
- If `sentence-transformers` is installed, a multilingual embedding model is used for dense retrieval; otherwise a TF-IDF+SVD (LSA) fallback keeps the notebook runnable offline.
- Set `CORPUS_DIR` to the folder of challenge Markdown files. If it is empty/missing, a small demo corpus is generated so every cell runs.

## 0. Configuration

In [1]:
from pathlib import Path
CORPUS_DIR   = Path("corpus_md")      # <- point to the challenge Markdown folder
BUILD_DIR    = Path("fieldfind_build")
MAX_TOKENS   = 350                    # max words per chunk
OVERLAP      = 50                     # word overlap when splitting long sections
DENSE_MODEL  = "intfloat/multilingual-e5-small"  # used only if sentence-transformers is available
USE_DENSE_MODEL = True                # set False to force the LSA fallback
TOP_K_CHUNKS = 100
RRF_K        = 60
BUILD_DIR.mkdir(exist_ok=True)

## 1. Imports (and optional install)

In [2]:
# Optional: !pip install sentence-transformers   (needs internet once to download the model)
import re, json, time, hashlib, math, collections
import numpy as np, pandas as pd
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
except ImportError:
    HAS_ST = False
print("sentence-transformers available:", HAS_ST)

sentence-transformers available: False


## 2. Demo corpus (only created if `CORPUS_DIR` has no .md files)

In [3]:
DEMO = {
 "cholera_response.md": """# Cholera Outbreak Response
## Case management
Patients with severe dehydration need IV Ringer lactate. Mild cases receive ORS (oral rehydration salts).
## Water treatment
Chlorination of drinking water at 0.5 mg/L free residual chlorine at the tap. Use HTH granules for bucket chlorination.
## Cholera treatment centre layout
Separate triage, observation and hospitalisation zones. Footbaths with 0.05% chlorine solution at entrances.""",
 "measles_vaccination.md": """# Measles Mass Vaccination Campaign
## Planning
Estimate target population aged 6 months to 15 years. Plan the cold chain: vaccines kept between +2 and +8 degrees C.
## Logistics
Order vaccines, auto-disable syringes and safety boxes. Prepare vaccination teams of 6 to 8 people.""",
 "fleet_management.md": """# Vehicle Fleet Management
## Fuel monitoring
Record fuel consumption per vehicle in the logbook. Compare litres per 100 km monthly.
## Security
Radio check-in every 2 hours on field movements. Never travel after dark without authorisation.""",
 "malnutrition_ctc.md": """# Severe Acute Malnutrition
## Admission criteria
MUAC below 115 mm or bilateral oedema. Children 6 to 59 months.
## Therapeutic feeding
Give F-75 in stabilisation phase, then F-100 or RUTF.""",
 "eau_potable_fr.md": """# Approvisionnement en eau potable
## Chloration
La chloration de l'eau de boisson vise un chlore résiduel libre de 0,5 mg/L. Utiliser du HTH.
## Stockage
Les réservoirs doivent être nettoyés et désinfectés chaque mois.""",
}
CORPUS_DIR.mkdir(exist_ok=True)
if not list(CORPUS_DIR.rglob("*.md")):
    for name, txt in DEMO.items():
        (CORPUS_DIR / name).write_text(txt, encoding="utf-8")
    print("Demo corpus written to", CORPUS_DIR)
print(len(list(CORPUS_DIR.rglob("*.md"))), "markdown files")

Demo corpus written to corpus_md
5 markdown files


## 3. Ingest & normalize → `docs`

In [4]:
def clean_md(t):
    t = re.sub(r"!\[[^\]]*\]\([^)]*\)", " ", t)          # image tags
    t = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", t)        # keep link text
    t = re.sub(r"<[^>]+>", " ", t)                         # html remnants
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def first_heading(t, fallback):
    m = re.search(r"^#\s+(.+)$", t, re.M)
    return m.group(1).strip() if m else fallback

rows = []
for p in sorted(CORPUS_DIR.rglob("*.md")):
    raw = p.read_text(encoding="utf-8", errors="ignore")
    txt = clean_md(raw)
    rows.append(dict(doc_id=hashlib.md5(str(p.relative_to(CORPUS_DIR)).encode()).hexdigest()[:10],
                     path=str(p.relative_to(CORPUS_DIR)), title=first_heading(txt, p.stem),
                     n_chars=len(txt), text=txt))
docs = pd.DataFrame(rows)
docs.drop(columns="text").head()

,doc_id,path,title,n_chars
0,a54a4e7178,cholera_response.md,Cholera Outbreak Response,435
1,9c8bb57923,eau_potable_fr.md,Approvisionnement en eau potable,219
2,080f828894,fleet_management.md,Vehicle Fleet Management,240
3,8acca60c45,malnutrition_ctc.md,Severe Acute Malnutrition,190
4,59a3c919ab,measles_vaccination.md,Measles Mass Vaccination Campaign,278


## 4. Section-aware chunking → `chunks`

In [5]:
def split_sections(text):
    """Yield (heading_path, body) using markdown headings."""
    path, buf, out = [], [], []
    for line in text.splitlines():
        m = re.match(r"^(#{1,6})\s+(.*)", line)
        if m:
            if buf: out.append((" > ".join(path), "\n".join(buf).strip()))
            lvl = len(m.group(1)); path = path[:lvl-1] + [m.group(2).strip()]; buf = []
        else:
            buf.append(line)
    if buf: out.append((" > ".join(path), "\n".join(buf).strip()))
    return [(h, b) for h, b in out if b]

def window(words, n, ov):
    if len(words) <= n: yield words; return
    for s in range(0, len(words), n - ov):
        yield words[s:s+n]
        if s + n >= len(words): break

crow = []
for d in docs.itertuples():
    for i, (h, body) in enumerate(split_sections(d.text) or [("", d.text)]):
        for j, w in enumerate(window(body.split(), MAX_TOKENS, OVERLAP)):
            crow.append(dict(chunk_id=f"{d.doc_id}_{i}_{j}", doc_id=d.doc_id, title=d.title,
                             heading=h, text=" ".join(w)))
chunks = pd.DataFrame(crow)
# text that is indexed: title + heading boost the body
chunks["index_text"] = chunks.title + " . " + chunks.heading + " . " + chunks.text
print(len(docs), "docs ->", len(chunks), "chunks")
chunks.head()

5 docs -> 11 chunks


,chunk_id,doc_id,title,heading,text,index_text
0,a54a4e7178_0_0,a54a4e7178,Cholera Outbreak Response,Cholera Outbreak Response > Case management,Patients with severe dehydration need IV Ringe...,Cholera Outbreak Response . Cholera Outbreak R...
1,a54a4e7178_1_0,a54a4e7178,Cholera Outbreak Response,Cholera Outbreak Response > Water treatment,Chlorination of drinking water at 0.5 mg/L fre...,Cholera Outbreak Response . Cholera Outbreak R...
2,a54a4e7178_2_0,a54a4e7178,Cholera Outbreak Response,Cholera Outbreak Response > Cholera treatment ...,"Separate triage, observation and hospitalisati...",Cholera Outbreak Response . Cholera Outbreak R...
3,9c8bb57923_0_0,9c8bb57923,Approvisionnement en eau potable,Approvisionnement en eau potable > Chloration,La chloration de l'eau de boisson vise un chlo...,Approvisionnement en eau potable . Approvision...
4,9c8bb57923_1_0,9c8bb57923,Approvisionnement en eau potable,Approvisionnement en eau potable > Stockage,Les réservoirs doivent être nettoyés et désinf...,Approvisionnement en eau potable . Approvision...


## 5. Lexical retriever: BM25 (pure numpy/scipy, no extra dependency)

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
import unicodedata

def norm(t):
    t = unicodedata.normalize("NFKD", t.lower())
    return "".join(c for c in t if not unicodedata.combining(c))   # strip accents (é -> e)

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
FR_STOP = set("le la les de des du un une et en au aux pour par sur dans est sont l d qu que qui avec ce cette".split())
STOP = sorted(ENGLISH_STOP_WORDS | FR_STOP)   # extend for other corpus languages

class BM25:
    def __init__(self, texts, k1=1.5, b=0.75):
        self.vec = CountVectorizer(preprocessor=norm, token_pattern=r"(?u)\b\w+\b", stop_words=STOP)
        tf = self.vec.fit_transform(texts).tocsc().astype(np.float32)
        N = tf.shape[0]; dl = np.asarray(tf.sum(1)).ravel(); avg = dl.mean()
        df = np.diff(tf.indptr)
        self.idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
        tf = tf.tocoo()
        denom = tf.data + k1 * (1 - b + b * dl[tf.row] / avg)
        w = tf.data * (k1 + 1) / denom * self.idf[tf.col]
        from scipy.sparse import csc_matrix
        self.W = csc_matrix((w, (tf.row, tf.col)), shape=tf.shape)
    def search(self, q, k=100):
        ids = [self.vec.vocabulary_[t] for t in self.vec.build_analyzer()(q) if t in self.vec.vocabulary_]
        if not ids: return []
        s = np.asarray(self.W[:, ids].sum(1)).ravel()
        top = np.argsort(-s)[:k]
        return [(int(i), float(s[i])) for i in top if s[i] > 0]

t0 = time.time(); bm25 = BM25(chunks.index_text.tolist()); print(f"BM25 built in {time.time()-t0:.2f}s")
bm25.search("chlorine water", 3)

BM25 built in 0.00s


[(1, 4.247910976409912), (2, 1.5111496448516846)]

## 6. Dense retriever (multilingual embeddings, LSA fallback)

In [7]:
class DenseST:
    name = "dense-st"
    def __init__(self, texts):
        self.m = SentenceTransformer(DENSE_MODEL, device="cpu")
        self.E = self.m.encode(["passage: " + t for t in texts], batch_size=64,
                               normalize_embeddings=True, show_progress_bar=True).astype(np.float16)
    def search(self, q, k=100):
        qv = self.m.encode(["query: " + q], normalize_embeddings=True)[0].astype(np.float32)
        s = self.E.astype(np.float32) @ qv
        top = np.argsort(-s)[:k]; return [(int(i), float(s[i])) for i in top]

class DenseLSA:
    """Offline fallback: char n-gram TF-IDF + SVD. Catches spelling/morphology variants."""
    name = "dense-lsa"
    def __init__(self, texts):
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        self.v = TfidfVectorizer(preprocessor=norm, analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
        X = self.v.fit_transform(texts)
        n = max(2, min(256, X.shape[0] - 1, X.shape[1] - 1))
        self.svd = TruncatedSVD(n, random_state=0); E = self.svd.fit_transform(X)
        self.E = (E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)).astype(np.float32)
    def search(self, q, k=100):
        qv = self.svd.transform(self.v.transform([q]))[0]; qv /= np.linalg.norm(qv) + 1e-9
        s = self.E @ qv; top = np.argsort(-s)[:k]; return [(int(i), float(s[i])) for i in top]

t0 = time.time()
dense = None
if HAS_ST and USE_DENSE_MODEL:
    try: dense = DenseST(chunks.index_text.tolist())
    except Exception as e: print("Model unavailable, falling back:", e)
if dense is None: dense = DenseLSA(chunks.index_text.tolist())
print(f"{dense.name} built in {time.time()-t0:.2f}s, matrix {dense.E.shape}")

dense-lsa built in 0.06s, matrix (11, 10)


## 7. Fusion, document grouping, confidence gate

In [8]:
def rrf(result_lists, k=RRF_K):
    sc = collections.defaultdict(float)
    for res in result_lists:
        for r, (i, _) in enumerate(res): sc[i] += 1 / (k + r + 1)
    return sorted(sc.items(), key=lambda x: -x[1])

def group_docs(chunk_scores, bonus=0.1):
    by = collections.defaultdict(list)
    for i, s in chunk_scores: by[chunks.doc_id.iat[i]].append((s, i))
    out = []
    for d, lst in by.items():
        lst.sort(reverse=True)
        score = lst[0][0] * (1 + bonus * min(len(lst) - 1, 3))   # best chunk + small bonus for support
        out.append((d, score, lst[0][1]))
    return sorted(out, key=lambda x: -x[1])

def search(q, mode="hybrid", k_docs=5, min_bm25=None):
    lex = bm25.search(q, TOP_K_CHUNKS) if mode in ("bm25", "hybrid") else []
    den = dense.search(q, TOP_K_CHUNKS) if mode in ("dense", "hybrid") else []
    fused = {"bm25": lambda: rrf([lex]), "dense": lambda: rrf([den]), "hybrid": lambda: rrf([lex, den])}[mode]()
    # confidence gate: require some lexical OR strong semantic evidence
    if min_bm25 is not None:
        best_lex = lex[0][1] if lex else 0.0
        best_den = den[0][1] if den else 0.0
        g_lex, g_den = min_bm25
        if best_lex < g_lex and best_den < g_den: return []
    lexset, denset = {i for i, _ in lex[:20]}, {i for i, _ in den[:20]}
    res = []
    for d, s, ci in group_docs(fused)[:k_docs]:
        res.append(dict(doc_id=d, title=chunks.title.iat[ci], section=chunks.heading.iat[ci],
                        score=round(s, 4), matched_by="+".join(m for m, S in (("kw", lexset), ("sem", denset)) if ci in S),
                        snippet=chunks.text.iat[ci][:180]))
    return res

pd.DataFrame(search("how to make drinking water safe with chlorine"))

,doc_id,title,section,score,matched_by,snippet
0,a54a4e7178,Cholera Outbreak Response,Cholera Outbreak Response > Water treatment,0.0393,kw+sem,Chlorination of drinking water at 0.5 mg/L fre...
1,9c8bb57923,Approvisionnement en eau potable,Approvisionnement en eau potable > Chloration,0.0175,sem,La chloration de l'eau de boisson vise un chlo...
2,59a3c919ab,Measles Mass Vaccination Campaign,Measles Mass Vaccination Campaign > Planning,0.0169,sem,Estimate target population aged 6 months to 15...
3,080f828894,Vehicle Fleet Management,Vehicle Fleet Management > Security,0.0167,sem,Radio check-in every 2 hours on field movement...
4,8acca60c45,Severe Acute Malnutrition,Severe Acute Malnutrition > Admission criteria,0.0164,sem,MUAC below 115 mm or bilateral oedema. Childre...


## 8. Interactive search helper

In [9]:
def show(q, mode="hybrid", k=5, gate=None):
    t0 = time.perf_counter(); r = search(q, mode, k, gate); ms = (time.perf_counter() - t0) * 1000
    print(f"Query: {q!r}  [{mode}]  {ms:.1f} ms")
    if not r: print("  No strong match in the collection."); return
    for n, x in enumerate(r, 1):
        print(f" {n}. {x['title']}  ›  {x['section']}   (score {x['score']}, {x['matched_by'] or 'fusion'})")
        print(f"    {x['snippet']}…")

show("ORS for dehydrated patients")
show("chloration eau")                 # French query
show("vaccine cold chain temperature")

Query: 'ORS for dehydrated patients'  [hybrid]  1.5 ms
 1. Cholera Outbreak Response  ›  Cholera Outbreak Response > Case management   (score 0.0393, kw+sem)
    Patients with severe dehydration need IV Ringer lactate. Mild cases receive ORS (oral rehydration salts).…
 2. Vehicle Fleet Management  ›  Vehicle Fleet Management > Security   (score 0.0172, sem)
    Radio check-in every 2 hours on field movements. Never travel after dark without authorisation.…
 3. Approvisionnement en eau potable  ›  Approvisionnement en eau potable > Stockage   (score 0.0169, sem)
    Les réservoirs doivent être nettoyés et désinfectés chaque mois.…
 4. Severe Acute Malnutrition  ›  Severe Acute Malnutrition > Admission criteria   (score 0.0164, sem)
    MUAC below 115 mm or bilateral oedema. Children 6 to 59 months.…
 5. Measles Mass Vaccination Campaign  ›  Measles Mass Vaccination Campaign > Planning   (score 0.0162, sem)
    Estimate target population aged 6 months to 15 years. Plan the cold chain: va

## 9. Evaluation harness
Edit `EVAL` with real challenge queries and the **paths** of relevant documents. `[]` = unanswerable query (should return nothing once the gate is on).
Optionally load from CSV: columns `query, relevant_paths` (semicolon-separated).

In [10]:
EVAL = [
 ("how to chlorinate drinking water",          ["cholera_response.md", "eau_potable_fr.md"]),
 ("ORS oral rehydration",                      ["cholera_response.md"]),
 ("keeping vaccines cold",                     ["measles_vaccination.md"]),
 ("MUAC admission threshold children",         ["malnutrition_ctc.md"]),
 ("fuel consumption of cars",                  ["fleet_management.md"]),
 ("chloration de l'eau",                       ["eau_potable_fr.md", "cholera_response.md"]),
 ("radio procedure during field movement",     ["fleet_management.md"]),
 ("satellite internet contract pricing",       []),
 ("tax declaration for expatriates",           []),
]
EVAL_CSV = Path("eval_queries.csv")
if EVAL_CSV.exists():
    e = pd.read_csv(EVAL_CSV); EVAL = [(r.query, [] if pd.isna(r.relevant_paths) else r.relevant_paths.split(";")) for r in e.itertuples()]

path2id = dict(zip(docs.path, docs.doc_id))

def evaluate(mode, gate=None, k=5):
    P, RR, R10, FP, lat = [], [], [], [], []
    for q, rel_paths in EVAL:
        rel = {path2id[p] for p in rel_paths if p in path2id}
        t0 = time.perf_counter(); r = search(q, mode, 10, gate); lat.append((time.perf_counter() - t0) * 1000)
        ids = [x["doc_id"] for x in r]
        if not rel: FP.append(len(ids[:k]) > 0); continue
        P.append(sum(i in rel for i in ids[:k]) / k)
        RR.append(next((1 / (n + 1) for n, i in enumerate(ids) if i in rel), 0))
        R10.append(len(rel & set(ids[:10])) / len(rel))
    return dict(mode=mode + ("+gate" if gate is not None else ""), **{"P@5": np.mean(P), "MRR": np.mean(RR),
                "Recall@10": np.mean(R10), "FP-rate unanswerable": np.mean(FP) if FP else np.nan,
                "p50 ms": np.median(lat), "p95 ms": np.percentile(lat, 95)})

# gate = (min BM25 score, min dense cosine); a query is rejected only if BOTH are below threshold.
# Grid search maximising MRR - FP-rate; ties -> loosest gate (keeps recall).
grid = [(gl, gd) for gl in [0, 1, 2, 3, 4, 6] for gd in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.01]]
def objective(g):
    r = evaluate("hybrid", g); return round(r["MRR"] - (r["FP-rate unanswerable"] or 0), 6)
best = max(grid, key=lambda g: (objective(g), -g[0], -g[1]))
print("Chosen gate (min_bm25, min_dense):", best)

results = pd.DataFrame([evaluate("bm25"), evaluate("dense"), evaluate("hybrid"), evaluate("hybrid", best)]).round(3)
results

Chosen gate (min_bm25, min_dense): (1, 0.8)


,mode,P@5,MRR,Recall@10,FP-rate unanswerable,p50 ms,p95 ms
0,bm25,0.200,1.000,0.857,0.0,0.284,0.342
1,dense,0.257,0.714,1.000,1.0,0.973,1.134
2,hybrid,0.257,1.000,1.000,1.0,1.164,1.201
3,hybrid+gate,0.257,1.000,1.000,0.0,1.145,1.190


Note: P@5 is capped low on a tiny corpus where most queries have only 1–2 relevant documents; MRR and Recall@10 are more informative there.
Calibrate the gate on a held-out set of queries for the final pitch to avoid overfitting.

## 10. Resource footprint & package bundle

In [11]:
import pickle, os
bundle = BUILD_DIR / "bundle"; bundle.mkdir(exist_ok=True)
chunks.drop(columns="index_text").to_pickle(bundle / "chunks.pkl")
docs.drop(columns="text").to_pickle(bundle / "docs.pkl")
pickle.dump(bm25, open(bundle / "bm25.pkl", "wb"))
np.save(bundle / "dense_vectors.npy", dense.E)
if isinstance(dense, DenseLSA): pickle.dump((dense.v, dense.svd), open(bundle / "lsa.pkl", "wb"))
manifest = dict(build_date=time.strftime("%Y-%m-%d %H:%M"), n_docs=len(docs), n_chunks=len(chunks),
                dense=dense.name, model=DENSE_MODEL if dense.name == "dense-st" else "tfidf-char-lsa",
                max_tokens=MAX_TOKENS, overlap=OVERLAP, gate=list(best),
                corpus_hash=hashlib.md5("".join(sorted(docs.doc_id)).encode()).hexdigest())
json.dump(manifest, open(bundle / "manifest.json", "w"), indent=2)
size = {f.name: f"{f.stat().st_size/1024:.1f} KB" for f in bundle.iterdir()}
print(json.dumps(manifest, indent=2)); size

{
  "build_date": "2026-10-28 13:16",
  "n_docs": 5,
  "n_chunks": 11,
  "dense": "dense-lsa",
  "model": "tfidf-char-lsa",
  "max_tokens": 350,
  "overlap": 50,
  "gate": [
    1,
    0.8
  ],
  "corpus_hash": "713dc0810aa223b8cd1d41652e2a3b49"
}


{'docs.pkl': '1.4 KB',
 'bm25.pkl': '7.4 KB',
 'lsa.pkl': '188.4 KB',
 'chunks.pkl': '3.1 KB',
 'manifest.json': '0.2 KB',
 'dense_vectors.npy': '0.6 KB'}

## Next steps
- Point `CORPUS_DIR` at the challenge Markdown, install `sentence-transformers`, re-run all cells.
- Replace `EVAL` with 30–50 labelled real queries; report the results table in the pitch.
- Optional: cross-encoder reranking of top-30 chunks, acronym glossary for query expansion, Streamlit front-end over `search()`.